# Train and inspect an Icescopy droplet model

This notebook uses the **five public labeled training images**, all **40 synthetic scenes**, and the separately distributed **General droplets 1.0.0** trainable checkpoint. It shows original labels, aligned training views, optional training, predictions on separate recordings, and export to an Icescopy model file.

**Run All is safe by default:** training, downloads, and model export are disabled. Data previews stay in memory. Evaluation writes overlays and a report into a new `results/notebook-…` directory. Source images, labels, and supplied models are never replaced.

The detector uses marked cells as examples of appearance **and** size. Training supplies one or two examples per view. The desktop app accepts any number of selected examples without retraining; with none, it uses a generic reference derived from the training data. More examples do not guarantee better detections.

## 1. Install and select a notebook environment

Use a separate **Python 3.11** environment. From the **repository root**, run:

```sh
python -m pip install -e ".[training]"
python -m pip install jupyterlab ipykernel
python -m ipykernel install --user --name icescopy-ml --display-name "Icescopy ML"
jupyter lab auto_cell_ml/train_and_evaluate.ipynb
```

Select the **Icescopy ML** kernel. For NVIDIA GPU training, install a compatible CUDA-enabled PyTorch build using the [official installation instructions](https://pytorch.org/get-started/locally/). Apple GPU training uses PyTorch's `mps` device; CPU training also works. No package installation or network download runs in the cells below.

Keep the repository layout intact: the training reader uses `src/`, and model export includes the license in `resources/models/`. The notebook accepts a kernel working directory at the repository root or `auto_cell_ml/`; it does not search other directories for missing data.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys
import zipfile
from collections import Counter
from datetime import datetime, timezone
import tempfile

cwd = Path.cwd().resolve()
if (cwd / "pyproject.toml").is_file() and (cwd / "auto_cell_ml/code").is_dir():
    REPO_ROOT, MODULE_ROOT = cwd, cwd / "auto_cell_ml"
elif cwd.name == "auto_cell_ml" and (cwd / "code").is_dir() and (cwd.parent / "pyproject.toml").is_file():
    REPO_ROOT, MODULE_ROOT = cwd.parent, cwd
else:
    raise RuntimeError("Start the notebook kernel in the repository root or auto_cell_ml/.")

os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
# Direct imports avoid a collision with Python's standard-library 'code' module.
sys.path.insert(0, str(MODULE_ROOT / "code"))
import numpy as np
import torch
import cv2
from PIL import Image, ImageDraw
from IPython.display import display, Markdown
import data
import model
import train
import evaluate
import export

REAL_MANIFEST = MODULE_ROOT / "examples/real/datasets.json"
SYNTHETIC_MANIFEST = MODULE_ROOT / "examples/synthetic/labels.json"
EVALUATION_EXAMPLES = MODULE_ROOT / "examples/real/evaluation/examples.json"
BUNDLED_CHECKPOINT = MODULE_ROOT / "models/general-droplets-1.0.0.pt"
BUNDLED_APP_MODEL = MODULE_ROOT / "models/general-droplets-1.0.0.icescopy-model"
print("Module:", MODULE_ROOT.relative_to(REPO_ROOT))
print("PyTorch:", torch.__version__, "| OpenCV:", cv2.__version__)

## 2. Verify the distributed files

The real-data manifest stores paths relative to itself, image hashes, recording identities, and saved sessions. The trusted reader loads the saved original frame and preserves every manual circle, including its radius. It also handles the original 16-bit images.

Training has **50 CSU cold-stage, 160 CSU IS PCR, 90 PKU, 16 TAMU, and 105 CIF circles**. Four setups have a separate, unlabeled evaluation recording; PKU has none. Another frame or crop from the same training recording is not an independent test.

A SHA-256 hash is a file fingerprint: a mismatch means the file differs from the recorded source. The check below verifies all supplied images and both model files before creating outputs. Reading evaluation files here checks their integrity; the training view generator never opens them.

In [ ]:
def local_file(base, relative):
    relative = Path(relative)
    if relative.is_absolute():
        raise ValueError(f"Expected a relative public path: {relative.name}")
    path = (base / relative).resolve()
    path.relative_to(base.resolve())  # Reject paths escaping the supplied data folder.
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

for path in (REAL_MANIFEST, SYNTHETIC_MANIFEST, EVALUATION_EXAMPLES,
             BUNDLED_CHECKPOINT, BUNDLED_APP_MODEL):
    if not path.is_file():
        raise FileNotFoundError(path)

real_manifest = json.loads(REAL_MANIFEST.read_text())
real_rows = real_manifest["setups"]
assert len(real_rows) == 5 and len({row["setup"] for row in real_rows}) == 5
real_sources = {}
for row in real_rows:
    image, circles, provenance = data._read_source(REAL_MANIFEST, row["setup"])
    real_sources[row["setup"]] = (image, circles, provenance)
    evaluation_row = row["evaluation"]
    if evaluation_row.get("image"):
        evaluation_path = local_file(REAL_MANIFEST.parent, evaluation_row["image"])
        assert data._hash(evaluation_path) == evaluation_row["sha256"]
        assert evaluation_row["recording_id"] != row["train"]["recording_id"]

synthetic_manifest = json.loads(SYNTHETIC_MANIFEST.read_text())
synthetic_rows = synthetic_manifest["scenes"]
assert Counter(row["split"] for row in synthetic_rows) == {"fit": 20, "validation": 10, "calibration": 10}
assert len({row["id"] for row in synthetic_rows}) == 40
for row in synthetic_rows:
    assert row["complete_labels"] is True
    image_path = local_file(SYNTHETIC_MANIFEST.parent, row["id"] + ".png")
    assert data._hash(image_path) == row["sha256"]
    with Image.open(image_path) as image:
        assert image.size == (row["width"], row["height"])

bundled_network, bundled_metadata = model.load_model(BUNDLED_CHECKPOINT, device="cpu")
with zipfile.ZipFile(BUNDLED_APP_MODEL) as bundle:
    app_metadata = json.loads(bundle.read("droplet_detector.json"))
    for file_key, hash_key in (("model_file", "sha256"), ("reference_model_file", "reference_sha256")):
        filename = app_metadata[file_key]
        assert Path(filename).name == filename
        assert hashlib.sha256(bundle.read(filename)).hexdigest() == app_metadata[hash_key]
assert app_metadata["version"] == bundled_metadata["version"] == "1.0.0"
assert app_metadata["model_id"] == bundled_metadata["model_id"] == "general-droplets"
VIEW_SIZE = int(bundled_metadata["crop_size"])
del bundled_network

lines = ["| Setup | Training circles | Separate evaluation recording |", "|---|---:|---|"]
for row in real_rows:
    lines.append(f"| {row['setup']} | {len(real_sources[row['setup']][1])} | {'Yes' if row['evaluation'].get('image') else 'None'} |")
display(Markdown("\n".join(lines)))
print("Verified 5 labeled real images, 4 separate images, 40 synthetic scenes, and both model files.")
print("Model:", app_metadata["name"], app_metadata["version"], "| training view:", VIEW_SIZE, "pixels")

## 3. Inspect all five original training images

Green circles are the saved user labels. Coordinates and radii use original image pixels; resizing below changes only the display. The model predicts circle centers and radii, rather than tracing each cell boundary.

The CIF training image is from Experiment 1 and the separate image is from Experiment 2 in the NC State CIF cold-stage example collection. Please retain its attribution and GPL-3.0 notice when redistributing it. See [examples/README.md](examples/README.md) for provenance and permissions for every setup.

Petters, Markus, & Yadav, Shweta (2023). *CIF-Cold-Stage/DropFreezingDetection.jl: v0.2.0*. Zenodo. [doi:10.5281/zenodo.7765097](https://doi.org/10.5281/zenodo.7765097). [Source release](https://github.com/CIF-Cold-Stage/DropFreezingDetection.jl/tree/v0.2.0).

In [ ]:
def circle_overlay(pixels, circles, *, color="#39ff76", width=2):
    preview = Image.fromarray(pixels).copy()
    draw = ImageDraw.Draw(preview)
    for x, y, radius in np.asarray(circles).reshape(-1, 3):
        draw.ellipse((x - radius, y - radius, x + radius, y + radius), outline=color, width=width)
    return preview

def contact_sheet(items, *, columns=3, width=400, height=330):
    sheet = Image.new("RGB", (columns * width, ((len(items) + columns - 1) // columns) * height), "white")
    draw = ImageDraw.Draw(sheet)
    for index, (title, image) in enumerate(items):
        x, y = (index % columns) * width, (index // columns) * height
        draw.text((x + 8, y + 8), title, fill="black")
        preview = image.copy()
        preview.thumbnail((width - 16, height - 42))
        sheet.paste(preview, (x + (width - preview.width) // 2, y + 34 + (height - 42 - preview.height) // 2))
    return sheet

items = []
for row in real_rows:
    pixels, circles, _ = real_sources[row["setup"]]
    items.append((f"{row['setup']} | {len(circles)} labels", circle_overlay(pixels, circles)))
display(contact_sheet(items))

## 4. Inspect synthetic targets and negatives

The procedural collection includes filled and frozen targets, empty holders, varied backgrounds, and lighting changes. **Green** marks targets and **red** marks explicitly rendered negative objects. Every other valid location is also a negative center target; an image region needs no separate negative annotation.

Only the **20 fit scenes** enter fresh training. The **10 validation** and **10 calibration** scenes remain separate. A synthetic scene declares `complete_labels: true` so that unmarked background can be treated as negative. These generated scenes help initialization; they do not prove performance on real recordings.

In [ ]:
fit_scenes = data.read_synthetic_fit(SYNTHETIC_MANIFEST)
assert len(fit_scenes) == 20
preview_rows = [next(row for row in synthetic_rows if row["split"] == split)
                for split in ("fit", "validation", "calibration")]
items = []
for row in preview_rows:
    with Image.open(local_file(SYNTHETIC_MANIFEST.parent, row["id"] + ".png")) as image:
        pixels = np.array(image.convert("RGB"))
    positives = [[c["x"], c["y"], c["radius"]] for c in row["targets"]]
    negatives = [[c["x"], c["y"], c["radius"]] for c in row["negatives"]]
    overlay = circle_overlay(pixels, positives)
    overlay = circle_overlay(np.array(overlay), negatives, color="#ef4444")
    items.append((f"{row['split']} | {len(positives)} targets, {len(negatives)} negatives", overlay))
display(contact_sheet(items))
print("Training reader opened only fit scenes; other splits above are visual previews.")

## 5. Preview aligned augmentation and complete source coverage

A training view is a square cut from the original image. Each epoch includes unrotated, scale-one tiles covering **every source pixel**, including outer background, plus a view of every labeled circle at a random interior position. Remaining views sample circles or background. The actual view count expands when these required views exceed the requested count. Training balances the setups to the largest view count.

Random views rotate through the full 360°, reflect, scale uniformly from 0.75–1.25, and modestly vary brightness, contrast, and gamma. Pixels, circles, and the valid-pixel mask receive the same geometric transformation. All circles remain in the returned labels, even outside the crop or partly crossing its edge. Only out-of-source padding is unknown.

The center grid has one location per 4×4 pixel block. Every fully valid block is negative unless it contains a marked center. Circle size is learned separately; the area inside a circle is not a filled positive mask. The two small reference slots share the query's orientation and lighting and retain their radius in query pixels. They come from the full source even when the query shows only background.

In [ ]:
SEED = 20261001
REQUESTED_VIEWS = 240
view_sets = {row["setup"]: data.TrainingViews(REAL_MANIFEST, row["setup"],
             size=VIEW_SIZE, count=REQUESTED_VIEWS, seed=SEED + index)
             for index, row in enumerate(real_rows)}
lines = ["| Setup | Required coverage tiles | Circle views | Actual views |", "|---|---:|---:|---:|"]
for setup, views in view_sets.items():
    lines.append(f"| {setup} | {views.tile_count} | {views.circle_count} | {len(views)} |")
display(Markdown("\n".join(lines)))

PREVIEW_SETUP = "02-CSU-IS-PCR-filled-and-empty"  # Change to any setup listed above.
views = view_sets[PREVIEW_SETUP]
background_index = next((index for index in range(views.tile_count)
                         if data.targets(*views[index][1:])["center"].sum() == 0), None)
indices = [0, views.tile_count, len(views) - 1]
if background_index is not None:
    indices.insert(1, background_index)
items = []
for index in dict.fromkeys(indices):
    pixels, circles, valid, references, mask, radii = views.episode(index)
    target = data.targets(circles, valid)
    positives = int(target["center"].sum())
    items.extend([(f"View {index} | {positives} centers", circle_overlay(pixels, circles)),
                  ("Valid pixels white; padding black", Image.fromarray(valid.astype(np.uint8) * 255).convert("RGB"))])
    for slot in range(2):
        title = f"Example {slot + 1} | radius {radii[slot]:.1f} query px" if mask[slot] else "Unused example slot"
        ref_circles = [[31.5, 31.5, 64 / 3]] if mask[slot] else []
        items.append((title, circle_overlay(references[slot], ref_circles)))
display(contact_sheet(items, columns=4, width=290, height=310))

# The same seed, epoch, and index reproduce pixels, labels, masks, and examples.
index = views.tile_count
first = views.episode(index)
assert all(np.array_equal(a, b) for a, b in zip(first, views.episode(index)))
views.set_epoch(1)
second = views.episode(index)
assert not np.array_equal(first[0], second[0])
views.set_epoch(0)
display(contact_sheet([("Epoch 0", circle_overlay(first[0], first[1])),
                       ("Epoch 1: a different aligned view", circle_overlay(second[0], second[1]))], columns=2))

## 6. Choose an optional training run

**Fine-tune** means starting from the supplied learned weights and fitting them to labeled data with a new optimizer. It needs no backbone download or access to the original private training run. The default below uses all five public setups; set `TRAINING_SETUPS` to a list of setup names to select fewer. To use your own completely marked sources, prepare the same manifest/session layout described in [README.md](README.md), change `TRAINING_MANIFEST`, and keep separate recordings for evaluation.

**Fresh training** starts from the official pretrained MobileNetV3-Small encoder, then trains on synthetic fit scenes and the selected real sources using one continuing model and optimizer. Set both `TRAINING_MODE = "fresh"` and `DOWNLOAD_BACKBONE = True` to allow its network download. The backbone comes from [TorchVision's documented ImageNet weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_small.html); its hash is checked.

An epoch is one pass through the balanced views. CPU runs can take longer; lower `REAL_EPOCHS` for an initial experiment. The trainer saves source hashes, settings, losses, and weights. Evaluation images and synthetic validation/calibration scenes never enter either training stage.

In [ ]:
RUN_TRAINING = False                 # Explicit opt-in; Run All does not fit weights.
TRAINING_MODE = "fine_tune"          # "fine_tune" or "fresh"
DOWNLOAD_BACKBONE = False            # Only used for opted-in fresh training.
TRAINING_MANIFEST = REAL_MANIFEST
TRAINING_SETUPS = None               # None = all marked setups; or ["03-PKU", ...].
REAL_EPOCHS = 20
SYNTHETIC_EPOCHS = 3                 # Fresh training only.
BATCH_SIZE = 8
SYNTHETIC_VIEWS = 64
THREADS = 4
ENCODER_LEARNING_RATE = 0.0002
HEAD_LEARNING_RATE = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
EVALUATION_DEVICE = "cpu"             # CPU gives an accessible default for visual evaluation.
THRESHOLD = 0.5

print("Training enabled:", RUN_TRAINING, "| mode:", TRAINING_MODE, "| available device:", DEVICE)

## 7. Run only when enabled

Every output goes into a new `auto_cell_ml/results/notebook-…` folder. A timestamp and unique suffix prevent collisions. Saving a stage twice in the same run raises an error instead of replacing its files. Restart the kernel and Run All to create a separate run.

The run manifest records input fingerprints and software versions. Training writes its more detailed `metadata.json` and `training.json` inside `training/`. The public `.pt` is a trainable checkpoint; the `.icescopy-model` is an inference bundle for the app. They serve different purposes.

In [ ]:
RUN_ROOT = None

def notebook_outputs():
    global RUN_ROOT
    if RUN_ROOT is None:
        results = MODULE_ROOT / "results"
        results.mkdir(exist_ok=True)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        RUN_ROOT = Path(tempfile.mkdtemp(prefix=f"notebook-{stamp}-", dir=results))
        provenance = {
            "notebook": "train_and_evaluate.ipynb", "created_utc": stamp,
            "real_manifest_sha256": data._hash(REAL_MANIFEST),
            "synthetic_manifest_sha256": data._hash(SYNTHETIC_MANIFEST),
            "evaluation_examples_sha256": data._hash(EVALUATION_EXAMPLES),
            "bundled_checkpoint_sha256": data._hash(BUNDLED_CHECKPOINT),
            "training_manifest_sha256": data._hash(TRAINING_MANIFEST),
            "seed": SEED, "training_enabled": RUN_TRAINING, "training_mode": TRAINING_MODE,
            "training_device": DEVICE, "evaluation_device": EVALUATION_DEVICE,
            "python": sys.version.split()[0], "torch": torch.__version__, "opencv": cv2.__version__,
            "numpy": np.__version__, "evaluation_used_for_training": False,
        }
        with (RUN_ROOT / "run.json").open("x") as stream:
            json.dump(provenance, stream, indent=2)
            stream.write("\n")
        print("New output folder:", RUN_ROOT.relative_to(MODULE_ROOT))
    return RUN_ROOT

trained_checkpoint = None
if RUN_TRAINING:
    if TRAINING_MODE not in ("fine_tune", "fresh"):
        raise ValueError("Choose fine_tune or fresh.")
    if TRAINING_MODE == "fresh" and not DOWNLOAD_BACKBONE:
        raise ValueError("Fresh training requires explicit DOWNLOAD_BACKBONE = True.")
    training_rows = json.loads(TRAINING_MANIFEST.read_text())["setups"]
    selected_names = train._select_setups(TRAINING_MANIFEST, TRAINING_SETUPS, TRAINING_SETUPS is None)
    independent_rows = [row["evaluation"] for row in real_rows if row["evaluation"].get("image")]
    for row in training_rows:
        if row["setup"] in selected_names:
            for independent in independent_rows:
                if (row["train"]["recording_id"] == independent["recording_id"]
                        or row["train"]["sha256"] == independent["sha256"]):
                    raise ValueError(f"Training/evaluation recording overlap: {row['setup']}")
    settings = dict(setup=TRAINING_SETUPS, all_setups=TRAINING_SETUPS is None,
                    epochs=REAL_EPOCHS, batch_size=BATCH_SIZE, seed=SEED, device=DEVICE,
                    threads=THREADS, size=VIEW_SIZE, views=REQUESTED_VIEWS,
                    synthetic_views=SYNTHETIC_VIEWS,
                    encoder_learning_rate=ENCODER_LEARNING_RATE, head_learning_rate=HEAD_LEARNING_RATE)
    output = notebook_outputs() / "training"
    if TRAINING_MODE == "fine_tune":
        trained_checkpoint = train.fit(TRAINING_MANIFEST, output,
            finetune_model=BUNDLED_CHECKPOINT, synthetic_epochs=0, **settings)
    else:
        from torch.hub import download_url_to_file
        backbone = notebook_outputs() / "mobilenet_v3_small-047dcff4.pth"
        if backbone.exists():
            raise FileExistsError(backbone)
        url = "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth"
        expected_hash = "047dcff4addef86ea5bc2eff13c9614dc11f47ab1160d0a71a25e7db994f4e1f"
        download_url_to_file(url, str(backbone), hash_prefix=expected_hash, progress=True)
        assert data._hash(backbone) == expected_hash
        trained_checkpoint = train.fit(TRAINING_MANIFEST, output,
            pretrained=backbone, synthetic_manifest=SYNTHETIC_MANIFEST,
            synthetic_epochs=SYNTHETIC_EPOCHS, **settings)
    print("New checkpoint:", trained_checkpoint.relative_to(MODULE_ROOT))
    history = json.loads((output / "training.json").read_text())
    print("Training history saved:", (output / "training.json").relative_to(MODULE_ROOT))
else:
    print("Training skipped. The supplied checkpoint will be evaluated below.")

## 8. Inspect predictions on separate recordings

Evaluation uses the newly trained checkpoint if one was created, otherwise the supplied public checkpoint. The provided guidance file contains one or two positive example circles per separate recording, selected independently of predictions. These examples guide appearance and size; they are **not complete evaluation labels**.

Green circles are new detections and cyan circles are supplied examples. Count differences are not accuracy scores. Missed cells, false selections, and incorrect radii require visual inspection. Complete independent reference labels would be needed to measure accuracy. The evaluator also saves labeled training-image diagnostics separately; those do not establish performance on new recordings. There is no independent PKU image.

`REAL_MANIFEST` and `EVALUATION_EXAMPLES` deliberately remain separate from any custom training manifest. Before training, the notebook rejects a selected training source whose recording identity or image fingerprint matches a public evaluation source. Keep this recording separation when preparing your own manifest.

In [ ]:
RUN_EVALUATION = True                # Inference only; writes a new output directory.
EVALUATION_MODEL = trained_checkpoint or BUNDLED_CHECKPOINT
report = None
if RUN_EVALUATION:
    evaluation_output = notebook_outputs() / "evaluation"
    report = evaluate.evaluate(REAL_MANIFEST, EVALUATION_EXAMPLES, EVALUATION_MODEL,
                              evaluation_output, device=EVALUATION_DEVICE, threshold=THRESHOLD)
    independent = [row for row in report["results"] if row["partition"] == "independent"]
    print("Separate recordings inspected:", len(independent))
    for row in independent:
        print(row["dataset"], "| new detections:", row["count"], "| examples:", len(row["examples"]))
    with Image.open(evaluation_output / "overview.jpg") as overview:
        display(overview.copy())
    print("Overlays and report:", evaluation_output.relative_to(MODULE_ROOT))
else:
    print("Evaluation skipped.")

## 9. Export a new version after training and review

Enable export only after training and inspecting the results. The exporter checks the model's recorded manifest hash and each training source before deriving the generic app reference. Supply a **new version** so the result remains distinguishable from General droplets 1.0.0.

Export produces the two small inference graphs, metadata, an `.icescopy-model` bundle, and a separate public `.pt` checkpoint. The public checkpoint omits private source paths and labels. The app bundle contains no training images or executable Python code. Its graph compatibility is checked before use.

To try the new file in Icescopy, open **Preferences → ML → Model file**. This selects a saved model; the app does not train online. Keep the original bundle and the full developer training metadata so results can be reproduced.

In [ ]:
RUN_EXPORT = False
MODEL_ID = "general-droplets"
MODEL_NAME = "General droplets"
NEW_VERSION = "1.0.1"

if RUN_EXPORT:
    if trained_checkpoint is None:
        raise RuntimeError("Export requires a new trained checkpoint from this run.")
    if NEW_VERSION == bundled_metadata["version"]:
        raise ValueError("Choose a version different from the supplied model.")
    root = notebook_outputs()
    archive = root / f"{MODEL_ID}-{NEW_VERSION}.icescopy-model"
    public_weights = root / f"{MODEL_ID}-{NEW_VERSION}.pt"
    export.export_bundle(trained_checkpoint, TRAINING_MANIFEST, root / "export",
                         version=NEW_VERSION, model_id=MODEL_ID, name=MODEL_NAME,
                         archive=archive, trainable=public_weights)
    sys.path.insert(0, str(REPO_ROOT / "src"))
    from icescopy_neural_detection import load_model as load_app_model, validate_model
    config = load_app_model(archive)
    validate_model(config)  # Fresh CPU graph checks; no training.
    print("Validated app bundle:", archive.relative_to(MODULE_ROOT))
    print("Separate trainable checkpoint:", public_weights.relative_to(MODULE_ROOT))
else:
    print("Export skipped.")

## What to preserve

Keep the run folder, the exact training manifest and original labeled sources, and the separate evaluation recordings. Review the independent overlays before choosing a new app model. Training losses and training-image matches describe the fitted data; neither replaces evaluation on new recordings.

See [README.md](README.md) for the command-line APIs and model contract, and [examples/README.md](examples/README.md) for public data attribution. Clear notebook outputs before committing an instructional copy if you do not intend to publish your run images or settings.